In [9]:
import json
from datetime import UTC, date, datetime
from pathlib import Path
from uuid import uuid4

In [10]:
import httpx

ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
DAILY_FIELDS = ["temperature_2m_max", "temperature_2m_min", "precipitation_sum"]

In [11]:
class TemporarySourceError(RuntimeError):
    """The source failed in a way that might work if we try again."""

In [12]:
def fetch_weather(latitude: float, longitude: float, start: date, end: date) -> dict:
    """Download daily weather for one place, from start to end (both included).

    Raises:
        TemporarySourceError: Network trouble, rate limiting or a server error.
        httpx.HTTPStatusError: Any other bad answer, such as a wrong parameter.
    """
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start.isoformat(),
        "end_date": end.isoformat(),
        "daily": ",".join(DAILY_FIELDS),
        "timezone": "UTC",
    }
    try:
        response = httpx.get(ARCHIVE_URL, params=params, timeout=30)
    except httpx.TransportError as error:
        raise TemporarySourceError(f"Could not reach Open-Meteo: {error}") from error
    if response.status_code == 429 or response.status_code >= 500:
        raise TemporarySourceError(f"Open-Meteo answered {response.status_code}. Try later.")
    response.raise_for_status()
    return response.json()

In [13]:
from weather_data_ml_pipeline.config import LOCATIONS

latitude, longitude = LOCATIONS['kumasi']

print(latitude, longitude)

bronze_data = fetch_weather(latitude, longitude, date(2026, 10, 1), date(2026, 11, 1))

print(bronze_data)

6.69 -1.62
{'latitude': 6.7135324, 'longitude': -1.5895691, 'generationtime_ms': 0.5679130554199219, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 259.0, 'daily_units': {'time': 'iso8601', 'temperature_2m_max': '°C', 'temperature_2m_min': '°C', 'precipitation_sum': 'mm'}, 'daily': {'time': ['2026-10-01', '2026-10-02', '2026-10-03', '2026-10-04', '2026-10-05', '2026-10-06', '2026-10-07', '2026-10-08', '2026-10-09', '2026-10-10', '2026-10-11', '2026-10-12', '2026-10-13', '2026-10-14', '2026-10-15', '2026-10-16', '2026-10-17', '2026-10-18', '2026-10-19', '2026-10-20', '2026-10-21', '2026-10-22', '2026-10-23', '2026-10-24', '2026-10-25', '2026-10-26', '2026-10-27', '2026-10-28', '2026-10-29', '2026-10-30', '2026-10-31', '2026-11-01'], 'temperature_2m_max': [28.7, 28.6, 30.7, 28.6, 28.4, 28.1, 29.7, 28.6, 30.5, 29.8, 30.2, 29.1, 29.3, 30.0, 30.2, 31.0, 30.9, 30.5, 31.0, 31.1, 31.1, 31.8, 30.9, 31.3, 31.3, 31.3, 29.8, 31.3, 31.5, 29.9, 31.0, 31.3], 

In [14]:
def save_bronze(payload: dict, location: str, bronze_dir: str) -> Path:
    """Save the response exactly as received, plus metadata about where it came from."""
    now = datetime.now(UTC)
    folder = Path(bronze_dir) / f"ingest_date={now.date()}" / f"location={location}"
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"{uuid4().hex}.json"
    record = {
        "_source": ARCHIVE_URL,
        "_location": location,
        "_ingested_at": now.isoformat(),
        "payload": payload,
    }
    path.write_text(json.dumps(record, indent=2))
    return path

In [15]:
save_bronze(bronze_data,'Kumasi','data/bronze')

WindowsPath('data/bronze/ingest_date=2026-11-28/location=Kumasi/f9f2a3fa6402400eac864cb26564a88d.json')

In [16]:
import polars as pl

FIELDS = {
    "time": "observed_on",
    "temperature_2m_max": "temp_max_c",
    "temperature_2m_min": "temp_min_c",
    "precipitation_sum": "precipitation_mm",
}
KEY = ["location", "observed_on"]

In [17]:
def validate_weather(payload: dict, location: str) -> pl.DataFrame:
    """Check the data contract and return clean Silver rows.

    Raises:
        ValueError: A field is missing or any row breaks a rule. One bad row rejects the batch.
    """
    daily = payload.get("daily")
    if not isinstance(daily, dict):
        daily = {}  # Treat a missing or malformed section as "every field is missing".
    missing = sorted(FIELDS.keys() - daily.keys())
    if missing:
        raise ValueError(f"{location}: missing fields: {', '.join(missing)}.")
    if len({len(daily[name]) for name in FIELDS}) != 1:
        raise ValueError(f"{location}: the fields have different numbers of values.")

    raw = pl.DataFrame({name: daily[name] for name in FIELDS}, strict=False)
    silver = raw.select(
        pl.lit(location).alias("location"),
        pl.col("time").cast(pl.String).str.to_date("%Y-%m-%d", strict=False).alias("observed_on"),
        *[
            pl.col(source).cast(pl.Float64, strict=False).alias(target)
            for source, target in FIELDS.items()
            if source != "time"
        ],
    )

    temperatures = pl.col("temp_max_c", "temp_min_c")
    rules = {
        "a value is missing or has the wrong type": pl.any_horizontal(pl.all().is_null()),
        "temperature is outside -90 to 60 °C": pl.any_horizontal(~temperatures.is_between(-90, 60)),
        "the minimum temperature is above the maximum": pl.col("temp_min_c") > pl.col("temp_max_c"),
        "precipitation is negative": pl.col("precipitation_mm") < 0,
        "the same location and date appear twice": pl.struct(KEY).is_duplicated(),
    }
    for problem, broken in rules.items():
        bad_rows = silver.filter(broken)
        if bad_rows.height:
            examples = bad_rows["observed_on"].head(3).to_list()
            raise ValueError(f"{location}: {problem} in {bad_rows.height} row(s), e.g. {examples}.")
    return silver.sort(KEY)

In [18]:
validate_weather(bronze_data,'kumasi')

location,observed_on,temp_max_c,temp_min_c,precipitation_mm
str,date,f64,f64,f64
"""kumasi""",2026-10-01,28.7,21.7,0.0
"""kumasi""",2026-10-02,28.6,20.5,0.0
"""kumasi""",2026-10-03,30.7,20.2,2.2
"""kumasi""",2026-10-04,28.6,22.0,8.8
"""kumasi""",2026-10-05,28.4,22.1,4.6
…,…,…,…,…
"""kumasi""",2026-10-28,31.3,22.6,2.8
"""kumasi""",2026-10-29,31.5,23.4,3.3
"""kumasi""",2026-10-30,29.9,22.9,2.6
